In [ ]:
from lightgbm import LGBMRegressor
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_validate
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

test = pd.read_csv('test.csv')


test['TotalSF'] = test['1stFlrSF'] + test['2ndFlrSF']+test['TotalBsmtSF']

test['HouseAge'] = test['YrSold'] - test['YearBuilt']

test['TotalBath'] = test['FullBath'] + test['HalfBath']*0.5 +test['BsmtFullBath']+test['BsmtHalfBath']*0.5


num_cols = test.select_dtypes(include='number').drop(columns=['Id']).columns
cat_cols = test.select_dtypes(exclude='number').columns

# 2. два маленьких пайплайна: по одному на группу столбцов
num_pipe = make_pipeline(SimpleImputer(strategy='median'))
cat_pipe = make_pipeline(
    SimpleImputer(strategy='constant', fill_value='None'),
    OneHotEncoder(handle_unknown='ignore'),
)

# 3. ColumnTransformer: (название, преобразователь, столбцы)
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols),
])


mask = (test['GrLivArea'] > 4000) & (test['SalePrice'] < 300000)
       

test_clean = test[~mask]
y_log_clean = np.log1p(train_clean['SalePrice'])
X_clean = test_clean.drop(columns=['Id'])

X_a = X_clean.copy()
X_a[['GarageYrBlt', 'MasVnrArea']] = X_a[['GarageYrBlt', 'MasVnrArea']].fillna(0)
print
model_lgbm = make_pipeline(preprocessor, LGBMRegressor(random_state=42, verbose=-1))
model_fit = model_lgbm.fit(X_a)

KeyError: 'SalePrice'